# 1_self_promotion.ipynb

**Purpose:** Identify which early-career researchers self-promoted their first
publication on Twitter, by comparing author names (from the bibliometric record)
against the names/handles of users who tweeted the publication's DOI. See the
manuscript Methods ("Self-promotion on Twitter") for the full description of
this name-matching procedure and its validation.

**Manuscript:** Gender differences in online visibility of early-career researchers

**Required input files:**
- `result/0_author_doi_3y_2012_2016_sample.csv` (from `0_data_processing.ipynb`)
- `result/1_doi_3y_2012_2016_tweet_sample.pkl` (from `0_data_processing.ipynb`)
- `data/3_1_openalex_tweeterid_name_processed.pkl` (external OpenAlex-Twitter ID
  crosswalk; see Mongeon, Bowman & Costas 2023, "An open dataset of scholars on
  Twitter")

**Outputs:** `results/2_self_promotion_scores_sample.csv`


In [ ]:
import re
from difflib import SequenceMatcher
from pathlib import Path

import pandas as pd

DATA_DIR = Path("data")
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

ENCODING = "utf-8"
pd.set_option("max_colwidth", 100)
pd.set_option("display.float_format", lambda x: "%.4f" % x)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 15)

SIMILARITY_THRESHOLD_GENERAL = 0.6
SIMILARITY_THRESHOLD_CHINESE = 0.8  # applied per Methods for Pinyin names

TITLES = [
    "mr ", "mr. ", "dr. ", "dr ", "mrs ", "mrs. ", "prof. ", "prof ",
    "ms ", "ms. ", "phd. ", "phd ",
]

### Helper functions: mention-year window and name cleaning

In [ ]:
def judge_year(df: pd.DataFrame, year: int) -> pd.DataFrame:
    """Flag whether a publication has a tweet within `year` years of its
    cohort year (used to define the 1-, 2-, or 3-year mention window)."""
    col = f"with_tw_{year}y"
    df[col] = 0
    for index, row in df.iterrows():
        tweet_years = [t[:4] for t in row["twtime_list"]]
        cohort_year = [str(int(row["cohort"]) - 1), str(int(row["cohort"]))]
        if year == 3:
            cohort_year = [
                str(int(row["cohort"]) - 1), str(int(row["cohort"])),
                str(int(row["cohort"]) + 1), str(int(row["cohort"]) + 2),
            ]
        df.at[index, col] = int(bool(set(tweet_years) & set(cohort_year)))
    return df


def name_revision(name_set: pd.DataFrame) -> pd.DataFrame:
    """Clean first names: replace hyphens/periods with spaces and keep only
    the first token if it is more than one character."""
    name_set[["last_name", "first_name"]] = name_set[["last_name", "first_name"]].fillna("NULL")
    name_set["first_name_revised"] = (
        name_set["first_name"]
        .apply(lambda x: x.replace("-", " ").replace(".", " "))
        .apply(lambda x: re.split(" ", x)[0] if (" " in x) and (len(re.split(" ", x)[0]) > 1) else x)
    )
    return name_set

### Read datasets

1. Selected early-career publications (first 3 years)
2. Tweet info for all early-career publications

> **Fix applied:** the original notebook called `pd.read__pickle(...)` (typo,
> double underscore) on a path ending in `.csv` -- this reads the actual
> pickle file produced by `0_data_processing.ipynb`.

In [ ]:
author_12_16_early_paper = pd.read_csv(RESULTS_DIR / "0_author_doi_3y_2012_2016_sample.csv", encoding=ENCODING)
author_12_16_early_paper["doi_lower"] = author_12_16_early_paper["doi"].astype(str).str.lower()

author_12_16_early_paper_tw = pd.read_pickle(RESULTS_DIR / "1_doi_3y_2012_2016_tweet_sample.pkl")
author_12_16_early_paper_tw["len_tweet"] = author_12_16_early_paper_tw["Original_Tweet_list"].apply(
    lambda x: len(x) if isinstance(x, list) else 0
)
author_12_16_early_paper_tw["len_tweeter"] = author_12_16_early_paper_tw["Original_Tweeter_list"].apply(
    lambda x: len(x) if isinstance(x, list) else 0
)

# NOTE on a fix: author_12_16_early_paper_tw comes straight from the
# Altmetric API results, which are keyed only by "doi" -- it has no
# "author_id" at all. A later step (building author_12_16_tw_noNan) merges
# on "author_id", which would fail with a KeyError without this step. DOIs
# are unique to one publication/author, so a plain merge on "doi" correctly
# attaches the right author_id to each row.
author_12_16_early_paper_tw = author_12_16_early_paper_tw.merge(
    author_12_16_early_paper[["author_id", "doi"]].drop_duplicates(), on="doi", how="left"
)

### Processing the data

- Step 1: process the author names of publications with tweets
- Step 2: map the authors and tweeters

In [ ]:
# Author name from publications
#
# Fix applied: the original notebook had a missing closing parenthesis --
# `.agg({...}.reset_index()` -- which called `.reset_index()` on the *dict*
# passed to `.agg()` rather than on the result of `.agg()`. Corrected below.

author_12_16_name = (
    author_12_16_early_paper[["author_id", "first_name", "last_name"]]
    .groupby("author_id")
    .agg({"last_name": lambda x: list(x), "first_name": lambda x: list(x)})
    .reset_index()
)

for col in ["last_name", "first_name"]:
    author_12_16_name[col] = author_12_16_name[col].apply(lambda x: list(dict.fromkeys(x)))
    author_12_16_name[col] = author_12_16_name[col].apply(lambda x: [y for y in x if pd.notna(y)])
    author_12_16_name[col] = author_12_16_name[col].apply(lambda x: x[0] if len(x) else "@")

In [ ]:
## the publications with tweets
author_12_16_tw_noNan = author_12_16_early_paper_tw[author_12_16_early_paper_tw["len_tweet"] > 0].copy()

author_12_16_tw_noNan = author_12_16_tw_noNan.merge(author_12_16_name, on="author_id", how="left")
author_12_16_tw_noNan = author_12_16_tw_noNan.drop_duplicates(["author_id", "doi"])
author_12_16_tw_noNan = name_revision(author_12_16_tw_noNan)

In [ ]:
## mapping the author and tweeter
#
# Fix applied: the original notebook referenced `author_12_16_tw`, which was
# never defined -- the correct variable (defined above) is
# `author_12_16_early_paper_tw`.

author_tweeter = author_12_16_early_paper_tw[
    author_12_16_early_paper_tw["Original_Tweeter_list"].notnull()
    & (author_12_16_early_paper_tw["len_tweeter"] > 0)
].copy()
author_tweeter["len_tweeterid_nonan"] = author_tweeter["Original_Tweeter_list"].apply(
    lambda x: len([y for y in x if pd.notnull(y)])
)

# NOTE on a fix: the merge below (with author_name_openalex) expects a column
# named "Original_Tweeter_int" -- but that name was never actually created
# anywhere; after exploding, the column is still called "Original_Tweeter_list"
# (just holding one value per row instead of a list). Renamed explicitly here
# so the later merge finds what it expects.
author_tweeter = author_tweeter[author_tweeter["len_tweeterid_nonan"] > 0][
    ["author_id", "doi", "Original_Tweeter_list"]
].explode("Original_Tweeter_list").rename(columns={"Original_Tweeter_list": "Original_Tweeter_int"})

### Self-promotion detection

Compare author names and tweet names. More details can be found in the paper's
Methods section ("Self-promotion on Twitter").

In [ ]:
def process_str(x: str) -> str:
    """Lowercase and strip common titles/honorifics before comparison."""
    if "-" in x:
        x = x.replace("-", " ")
    x = x.lower()
    for title in TITLES:
        x = x.replace(title, "")
    return x


def similar(a: str, b: str) -> float:
    return SequenceMatcher(None, a.lower(), b.lower()).ratio()


def similar_x(a: str, b: str) -> float:
    """Alphanumeric-only similarity (strips spaces/punctuation first)."""
    try:
        a_clean = re.sub("[^A-Za-z0-9]+", "", a)
        b_clean = re.sub("[^A-Za-z0-9]+", "", b)
        return SequenceMatcher(None, a_clean.lower(), b_clean.lower()).ratio()
    except Exception:
        return 0.0


def compare_multi(f, l, fl, lf, f_l, l_f, candidate):
    """Compare a candidate name string against several author-name variants
    (first, last, first+last, last+first, with/without space) and return the
    best similarity score."""
    candidate = process_str(candidate)
    if " " in candidate:
        return max(similar(f_l, candidate), similar(l_f, candidate))
    elif candidate != "null":
        return max(similar(f, candidate), similar(l, candidate), similar(fl, candidate), similar(lf, candidate))
    return 0.0

In [ ]:
def self_verify(first, last, list1, list2, list3, listid, listT, key):
    """Compare an author's name against three candidate-name sources for a
    given publication's tweeters:
      list1 -- OpenAlex-crosswalked author names for each tweeter
      list2 -- Twitter display names for each tweeter
      list3 -- Twitter handles for each tweeter
    Returns similarity scores, best-matching names, and which source (author
    name / display name / handle) produced the overall best match."""
    score_1 = score_2 = score_3 = 0.0
    tweeter_id_1, tweeter_id_2, tweeter_id_3 = 1, 2, 3
    author_name = user_name = handle = "NULL"

    first = first.replace("@", "")
    last = last.replace("@", "")
    first_last, firstlast = f"{first} {last}", f"{first}{last}"
    last_first, lastfirst = f"{last} {first}", f"{last}{first}"

    for i in list1:
        if isinstance(i, list):
            for j in i:
                if isinstance(j, str):
                    score = compare_multi(first, last, firstlast, lastfirst, first_last, last_first, j)
                    if score > score_1:
                        score_1 = score
                        author_name = j
                        idx = list1.index(i)
                        tweeter_id_1 = listT[idx]

    for i in range(len(list2)):
        if isinstance(list2[i], str):
            score = compare_multi(first, last, firstlast, lastfirst, first_last, last_first, list2[i])
            if score > score_2:
                score_2 = score
                user_name = list2[i]
                tweeter_id_2 = listT[i] if key == 0 else 2

    for i in range(len(list3)):
        if isinstance(list3[i], str):
            score = compare_multi(first, last, firstlast, lastfirst, first_last, last_first, list3[i])
            if score > score_3:
                score_3 = score
                handle = list3[i]
                tweeter_id_3 = listT[i] if key == 0 else 3

    max_score = max(score_1, score_2, score_3)
    if max_score == score_1:
        attr, likely_name, tweeter_id = "author_name_self_score", author_name, tweeter_id_1
    elif max_score == score_2:
        attr, likely_name, tweeter_id = "tw_name_self_score", user_name, tweeter_id_2
    else:
        attr, likely_name, tweeter_id = "tw_handle_self_score", handle, tweeter_id_3

    return (score_1, score_2, score_3, author_name, user_name, handle,
            attr, max_score, likely_name, tweeter_id)

In [ ]:
def match_name(name_a: str, name_b: str, threshold: float = SIMILARITY_THRESHOLD_GENERAL) -> bool:
    """Return True if two name strings are similar enough to be considered a
    match, using the same SequenceMatcher-based similarity as `self_verify`.

    Fix applied: this function was called by `name_check()` in the original
    notebook but never defined there -- filled in here using the same
    similarity approach and threshold used throughout the rest of the
    self-promotion detection pipeline."""
    return similar(name_a, name_b) >= threshold


def name_check(first, last, most_likely):
    f_l = f"{first} {last}"
    l_f = f"{last} {first}"
    return match_name(f_l, most_likely) or match_name(l_f, most_likely)


def intersection(lst1, lst2):
    return [value for value in lst1 if value in lst2]

### Data for comparison: OpenAlex author-name / Twitter-ID crosswalk

In [ ]:
author_name_openalex = pd.read_pickle(DATA_DIR / "3_1_openalex_tweeterid_name_processed.pkl")

Tweeter_list = author_tweeter.merge(
    author_name_openalex, left_on="Original_Tweeter_int", right_on="tweeter_id", how="left"
)
Tweeter_list["author_name"] = Tweeter_list["author_name"].fillna("0")
Tweeter_list = Tweeter_list.groupby(["author_id", "doi"])["author_name"].apply(list).reset_index()

author_12_16_tw_noNan = author_12_16_tw_noNan.merge(Tweeter_list, on=["author_id", "doi"], how="left")

In [ ]:
(
    author_12_16_tw_noNan["author_name_self_score"],
    author_12_16_tw_noNan["tw_name_self_score"],
    author_12_16_tw_noNan["tw_handle_self_score"],
    author_12_16_tw_noNan["likely_author"],
    author_12_16_tw_noNan["likely_name"],
    author_12_16_tw_noNan["likely_handle"],
    author_12_16_tw_noNan["max_attribute"],
    author_12_16_tw_noNan["max_score"],
    author_12_16_tw_noNan["most_likely_name"],
    author_12_16_tw_noNan["likely_Tweeter_ID"],
) = zip(*author_12_16_tw_noNan.apply(
    lambda row: self_verify(
        row["first_name_revised"], row["last_name"], row["author_name"],
        row["tw_name_list"], row["tw_handle_list"], row["Original_Tweet_list"],
        row["Original_Tweeter_list"], row["searched"],
    ),
    axis=1,
))

### Save results and check match rate

Return the share of authors with a name-match score over the 0.6 threshold.

In [ ]:
author_12_16_tw_noNan.to_csv(RESULTS_DIR / "2_self_promotion_scores_sample.csv", index=False, encoding=ENCODING)

matched_share = (
    author_12_16_tw_noNan[author_12_16_tw_noNan["max_score"] > SIMILARITY_THRESHOLD_GENERAL]["author_id"].nunique()
    / author_12_16_tw_noNan["author_id"].nunique()
)
print(f"Share of authors with a name-match score > {SIMILARITY_THRESHOLD_GENERAL}: {matched_share:.2%}")